# Data Cleaning — Used Car Price Prediction

Executes the cleaning decisions documented in `01_eda.ipynb`, as functions from
`src/data_prep.py`. Runs on the whole dataset — every step here is deterministic
(unit parsing, a known-bad-row removal, domain-knowledge flags), not statistical,
so nothing here risks leaking train-only information. Imputation and encoding are
NOT done here; they depend on training-set statistics and belong in the
preprocessing Pipeline, built after the train/test split.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.data_prep import (
    load_raw_data, drop_index_column, drop_duplicate_rows,
    remove_implausible_kilometers, parse_engine, parse_power,
    parse_mileage, resolve_new_price, clean_data
)

pd.set_option('display.max_columns', None)

In [2]:
df = load_raw_data("../data/raw/train-data.csv")
df.shape

(6019, 14)

In [3]:
df = drop_index_column(df)
assert 'Unnamed: 0' not in df.columns
df.columns.tolist()

['Name',
 'Location',
 'Year',
 'Kilometers_Driven',
 'Fuel_Type',
 'Transmission',
 'Owner_Type',
 'Mileage',
 'Engine',
 'Power',
 'Seats',
 'New_Price',
 'Price']

In [4]:
df = drop_duplicate_rows(df)
df.shape

Dropped 0 duplicate row(s).


(6019, 13)

In [5]:
df = remove_implausible_kilometers(df)
assert df['Kilometers_Driven'].max() < 1_000_000
df['Kilometers_Driven'].describe()

Removed 1 row(s) with Kilometers_Driven >= 1,000,000.


count      6018.000000
mean      57668.047690
std       37878.783175
min         171.000000
25%       34000.000000
50%       53000.000000
75%       73000.000000
max      775000.000000
Name: Kilometers_Driven, dtype: float64

In [6]:
df = parse_engine(df)
assert df['Engine'].dtype == 'float64'
df['Engine'].describe()

count    5982.000000
mean     1621.047141
std       601.143848
min        72.000000
25%      1198.000000
50%      1493.000000
75%      1984.000000
max      5998.000000
Name: Engine, dtype: float64

In [7]:
df = parse_power(df)
assert df['Power'].dtype == 'float64'
df['Power'].isnull().sum()  # includes original NaNs + former "null bhp" rows now correctly NaN

np.int64(143)

In [8]:
df = parse_mileage(df)
df[['Mileage_value', 'Mileage_unit']].head()

,Mileage_value,Mileage_unit
0,26.60,km/kg
1,19.67,kmpl
2,18.20,kmpl
3,20.77,kmpl
4,15.20,kmpl


In [9]:
df['Mileage_unit'].value_counts(dropna=False)

Mileage_unit
kmpl     5950
km/kg      66
NaN         2
Name: count, dtype: int64

In [10]:
df = resolve_new_price(df)
assert 'New_Price' not in df.columns
df['Had_New_Price'].value_counts()

Had_New_Price
0    5194
1     824
Name: count, dtype: int64

In [11]:
df_check = clean_data(load_raw_data("../data/raw/train-data.csv"))
pd.testing.assert_frame_equal(
    df.reset_index(drop=True), df_check.reset_index(drop=True)
)
print("clean_data() reproduces the step-by-step result exactly.")

Dropped 0 duplicate row(s).
Removed 1 row(s) with Kilometers_Driven >= 1,000,000.
clean_data() reproduces the step-by-step result exactly.


This check matters beyond just "does it work" — it confirms `clean_data()` is a
faithful, reusable stand-in for these steps, which is what lets `03_feature_engineering`
(and eventually the test suite) call one function instead of repeating this notebook.

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6018 entries, 0 to 6017
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Name               6018 non-null   str    
 1   Location           6018 non-null   str    
 2   Year               6018 non-null   int64  
 3   Kilometers_Driven  6018 non-null   int64  
 4   Fuel_Type          6018 non-null   str    
 5   Transmission       6018 non-null   str    
 6   Owner_Type         6018 non-null   str    
 7   Engine             5982 non-null   float64
 8   Power              5875 non-null   float64
 9   Seats              5976 non-null   float64
 10  Price              6018 non-null   float64
 11  Mileage_value      6016 non-null   float64
 12  Mileage_unit       6016 non-null   str    
 13  Had_New_Price      6018 non-null   int64  
dtypes: float64(5), int64(3), str(6)
memory usage: 658.3 KB


In [14]:
df.to_csv("../data/processed/used_cars_cleaned.csv", index=False)
print(f"Saved cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Saved cleaned dataset: 6018 rows, 14 columns
